# p- vs h-Refinement Demonstration

Canonical data come from the package CLI. Regenerate the CSV with:

```bash
packages/stenotic-hemodynamics/bin/stenotic-hemodynamics verify ph-refinement \
  --h-nxs 20,40,80,160 \
  --h-degree 2 \
  --degrees 0,1,2,3,4 \
  --p-nx 40 \
  --output-dir report/assets/data/verification \
  --summary-tex report/assets/tables/verification/p_h_refinement_demo.tex \
  --overwrite
```

Then regenerate report assets with:

```bash
pipenv run ops-render-ph-refinement-demo
```

In [ ]:
import csv
from pathlib import Path

import matplotlib.pyplot as plt

def as_float(row, key):
    value = row.get(key, "")
    return float(value) if value else float("nan")

csv_path = Path("report/assets/data/verification/p_h_refinement_demo.csv")
with csv_path.open(newline="") as handle:
    rows = [row for row in csv.DictReader(handle) if row["status"].lower() == "ok"]
rows

In [ ]:
h_rows = sorted([row for row in rows if row["sweep"] == "h_refinement"], key=lambda row: as_float(row, "dofs"))
p_rows = sorted([row for row in rows if row["sweep"] == "p_refinement"], key=lambda row: as_float(row, "degree"))

fig, (ax_cost, ax_degree) = plt.subplots(1, 2, figsize=(8, 3.5))

ax_cost.loglog([as_float(row, "dofs") for row in h_rows], [as_float(row, "area_l2_error") for row in h_rows], marker="o", label="h-refinement, fixed p=2")
ax_cost.loglog([as_float(row, "dofs") for row in p_rows], [as_float(row, "area_l2_error") for row in p_rows], marker="s", label="p-refinement, fixed mesh")
ax_cost.set_xlabel("State degrees of freedom")
ax_cost.set_ylabel("Area L2 error")
ax_cost.set_title("Error versus cost")
ax_cost.grid(True, which="both", alpha=0.25)
ax_cost.legend(frameon=False)

ax_degree.plot([as_float(row, "degree") for row in p_rows], [as_float(row, "area_log10_l2_error") for row in p_rows], marker="s")
ax_degree.set_xlabel("Polynomial degree p")
ax_degree.set_ylabel("log10 area L2 error")
ax_degree.set_title("Fixed-mesh p sweep")
ax_degree.grid(True, alpha=0.25)

fig.tight_layout()